## list tools

In [98]:

import asyncio
import json
from fastmcp import Client
import os
import json
from fastmcp.client.transports import StreamableHttpTransport

PORT = os.getenv("PORT", "9000")
MCP_PATH = os.getenv("MCP_PATH", "/mcp/knowledge")

async def example_list_tools():
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
        headers={"x-forwarded-user": "123"},
    )) as client:
        await client.ping()
        
        # List available operations
        tools = await client.list_tools()
        for tool in tools:
            print(tool.name)

await example_list_tools()


create_knowledge_set
list_knowledge_sets
delete_knowledge_set
ingest_file
list_files
remove_file
query


# Management

## list knowledge set

In [99]:
async def example_list_tenants():
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
    )) as client:
        await client.ping()
        tenants = await client.call_tool(
            name="list_knowledge_sets",
            arguments={},
        )
        if tenants:
            return json.loads(tenants[0].text)
        else:
            return []


list_tenants = await example_list_tenants()
print(json.dumps(list_tenants, indent=4))

[
    {
        "tenant_id": "b4047d20-54ba-4954-909e-2ec1e74189c9",
        "created_at": "2025-07-02T23:53:24.646408Z"
    },
    {
        "tenant_id": "9f3086be-6d9b-4073-a194-c92b8685fe4b",
        "created_at": "2025-07-02T23:53:28.195923Z"
    },
    {
        "tenant_id": "af8dfea7-148f-42c9-ad2d-cc4e8a522764",
        "created_at": "2025-07-02T23:53:29.412984Z"
    }
]


## create knowledge set

In [100]:
async def example_create_tenant():
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
    )) as client:
        await client.ping()
        tenants = await client.call_tool(
            name="create_knowledge_set",
            arguments={
            },
        )
        print("created tenant")
        print(tenants[0].text)
        return json.loads(tenants[0].text)["tenant_id"]

tenant_id = await example_create_tenant()

created tenant
{
  "tenant_id": "050472fe-97c1-4753-b453-f59c31f9cd9f",
  "created_at": "2025-07-02T17:38:13.582678"
}


## delete knowledge set

In [101]:
async def example_delete_tenant():
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
        headers={"x-forwarded-user": tenant_id},
    )) as client:
        await client.ping()
        tenants = await client.call_tool(
            name="delete_knowledge_set",
            arguments={
            },
        )
        print(f"deleted tenant {tenant_id}")
        print(tenants[0].text)

await example_delete_tenant()

deleted tenant 050472fe-97c1-4753-b453-f59c31f9cd9f
{
  "detail": "tenant and related data deleted"
}


In [102]:
list_tenants = await example_list_tenants()
list_tenants

[{'tenant_id': 'b4047d20-54ba-4954-909e-2ec1e74189c9',
  'created_at': '2025-07-02T23:53:24.646408Z'},
 {'tenant_id': '9f3086be-6d9b-4073-a194-c92b8685fe4b',
  'created_at': '2025-07-02T23:53:28.195923Z'},
 {'tenant_id': 'af8dfea7-148f-42c9-ad2d-cc4e8a522764',
  'created_at': '2025-07-02T23:53:29.412984Z'}]

# Knowledge Ingest, query, remove


In [103]:
test_tenant_id = list_tenants[-1]["tenant_id"]
test_tenant_id

'af8dfea7-148f-42c9-ad2d-cc4e8a522764'

## list files

In [104]:
async def example_list_files():
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
        headers={"x-forwarded-user": test_tenant_id},
    )) as client:
        await client.ping()
        files = await client.call_tool(
            name="list_files",
            arguments={},
        )
        if files:
            return json.loads(files[0].text)
        else:
            return []

await example_list_files()

[]

## Ingest file

In [105]:
async def example_ingest_file(file_path: str):
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
        headers={"x-forwarded-user": test_tenant_id},
    )) as client:
        await client.ping()
        with open(file_path, "rb") as f:
            content = f.read()
        files = await client.call_tool(
            name="ingest_file",
            arguments={
                "filename": file_path.split("/")[-1],
                "content_type": "text/plain",
                "content": content
            },
        )
        if files:
            return json.loads(files[0].text)

In [111]:
file_id_acorn = await example_ingest_file("test_acorn.txt")
file_id_acorn = file_id_acorn["file_id"]
file_id_acorn


'abede9fa-c9e0-46d0-a2d6-87359477045b'

In [112]:
file_id_fastmcp = await example_ingest_file("test_fastmcp.txt")
file_id_fastmcp = file_id_fastmcp["file_id"]
file_id_fastmcp

'97ae43dd-079c-48ba-a0e6-f7e56e10ba85'

## Remove File

In [108]:
async def example_remove_file(file_id: str):
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
        headers={"x-forwarded-user": test_tenant_id},
    )) as client:
        await client.ping()
        await client.call_tool(
            name="remove_file",
            arguments={"file_id": file_id},
        )
        print("file removed")

In [109]:
await example_remove_file(file_id_fastmcp)

file removed


In [110]:
await example_remove_file(file_id_acorn)

file removed


In [97]:
file_lists = await example_list_files()
file_lists

[]

## Query

In [113]:
async def example_query(query_text: str):
    async with Client(transport=StreamableHttpTransport(
        f"http://127.0.0.1:{PORT}{MCP_PATH}",
        headers={"x-forwarded-user": test_tenant_id},
    )) as client:
        await client.ping()
        query_result = await client.call_tool(
            name="query",
            arguments={"query_text": query_text},
        )
        return json.loads(query_result[0].text)

In [114]:
query_text = "tell me something about acorn labs?"
res = await example_query(query_text)
res

[{'file_id': 'abede9fa-c9e0-46d0-a2d6-87359477045b',
  'chunk_id': 'abede9fa-c9e0-46d0-a2d6-87359477045b_chunk_1',
  'score': 0.36180049435500494,
  'metadata': {'text': 'urce projects, we are committed to transparency and supporting a thriving open source ecosystem Join Our Early-Stage Startup At Acorn Labs, we’re building a team where amazing people can do their best work. Our team is a close-knit group of professionals who believe in the power of collaboration, treating everyone with the utmost respect, and finding joy in every step of our journey. We’re not afraid to tackle complex problems head-on, yet we never lose sight of the importance of a good laugh and not taking ourselves too seriously. Ready to Join the Acorn Team? If you’re eager to be a part of our dynamic team and embark on an exciting career path, we invite you to send us your resume along with a brief introduction that tells us more about yourself. Please feel free to reach out to us by emailing your resume to career

In [115]:
query_text = "how does fastmcp work?"
res = await example_query(query_text)
res

[{'file_id': '97ae43dd-079c-48ba-a0e6-f7e56e10ba85',
  'chunk_id': '97ae43dd-079c-48ba-a0e6-f7e56e10ba85_chunk_3',
  'score': 0.3250081956367026,
  'metadata': {'text': 'complex protocol details and server management, so you can focus on building great tools. It’s designed to be high-level and Pythonic; in most cases, decorating a function is all you need. FastMCP 2.0 has evolved into a comprehensive platform that goes far beyond basic protocol implementation. While 1.0 provided server-building capabilities (and is now part of the official MCP SDK), 2.0 offers a complete ecosystem including client libraries, authentication systems, deployment tools, integrations with major AI platforms, testing frameworks, and production-ready infrastructure patterns. FastMCP aims to be: 🚀 Fast: High-level interface means less code and faster development 🍀 Simple: Build MCP servers with minimal boilerplate 🐍 Pythonic: Feels natural to Python developers 🔍 Complete: A comprehensive platform for all MCP u